## AdaGrad (Adaptive Gradient)

During the training process, some weights can rise significantly, while others tend to not change much. AdGrad institutes a pre-parameter learning rate rather than a globally shared-rate. It provide a way to normalize parameter updates by keeping a history of previous updates. The bigger the sum of the updates is, in either direction, the smaller updates are made further in training.

* This will let less-frequently updates paramters to keep-up with the changes, effectively utilizing more neurons for training.

In [ ]:
# cache += parm_gradient ** 2
# parm_updates = learning_rate * parm_gradient / (sqrt(cache) + eps)

* cache holds the history.
* epsilon is a hyperparameter preventing division by 0.
* We are taking the square, so that we do not divide by a negative number. Also taking square and then taking square root will make cache grow slower.
* The division performed with a constantly rising cache might also cause learning to stall as updates become smaller with time.

SGD typically takes a longer route to reach the minimum even if there is only one minimum, it first changes in the direction of the weight which is changing a lot, and neuron weights will smaller change do not contribute much to the direction. 

Adagrad solves this problem, as if one weights has changes much, the gradient decreases for that parameter focusing on other parameters that haven't changed much. Adagrad hence takes a smaller, much quicker route to the minimum.


#### Drawbacks of Adagrad:-

* Adagrad can only decrease the gradient and not increase it. Hence if the target is still far away, and even though a lot of progress happened in one parameter but a lot of progress is still left, the Adagrad will still scale down the gradient.



In [ ]:
import numpy as np
from nnfs.datasets import vertical_data, spiral_data
import nnfs

In [ ]:
class Optimizer_Adagrad:

    def __init__(self, learning_rate = 1.0, decay = 0, epsilon = 1e-7):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay = decay
        self.iterations = 0
        self.epsilon = epsilon

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate * (1/(1+ self.decay * self.iterations))


    #update parameters
    def update_params(self, layer):

        if not hasattr(layer, 'weight_cache'):
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_cache = np.zeros_like(layer.biases)

        # Update cache with squared current gradients
        layer.weight_cache += layer.dweights ** 2
        layer.bias_cache += layer.dbiases ** 2

        layer.weights += -self.current_learning_rate * layer.dweights / (np.sqrt(layer.weight_cache) + self.epsilon)
        layer.biases += -self.current_learning_rate * layer.dbiases / (np.sqrt(layer.bias_cache) + self.epsilon)

    def post_update_params(self):
        self.iterations += 1

In [ ]:
class Dense_layer:

    def __init__(self, n_inputs, n_neurons):
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))
    
    # we want to remember our input to calculate gradient of weights
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.dot(inputs, self.weights) + self.biases

    
    def backward(self, dvalues):
        self.dweights = np.dot(self.inputs.T, dvalues)
        self.dbiases = np.sum(dvalues, axis = 0, keepdims = True)
        self.dinputs = np.dot(dvalues, self.weights.T)


class Activation_Relu:

    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)

    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] = 0


class Loss():
    def calculate(self, output, y):
        #calculate sample losses 
        sample_losses = self.forward(output, y)

        data_loss = np.mean(sample_losses)

        return data_loss
    

class Loss_CategoricalCrossentropy(Loss):

    def forward(self, y_pred, y_true):

        samples = len(y_pred)

        #We are capping the y_pred for preventing log(0) or log(high value) both are not defined
        y_pred_clipped = np.clip(y_pred, 1e-7, 1- 1e-7)

        if len(y_true.shape) == 1:
            correct_confidences = y_pred_clipped[range(samples), y_true]

        if len(y_true.shape) == 2:
            correct_confidences = np.sum(y_pred_clipped * y_true, axis = 1)

        negative_log_likelihood = -np.log(correct_confidences)
        return negative_log_likelihood
    
    def backward(self, dvalues, y_true):
        samples = len(dvalues) # Number of samples

        #Number of labels in every sample
        #We'll use the first sample to count them
        labels = len(dvalues[0])

        #If labels are sparse then, turn then into one-hot vector
        if len(y_true.shape) == 1:
            y_true =   np.eye(labels)[y_true]

        #calculate gradient
        self.dinputs = -y_true / dvalues

        #Normalize gradient
        self.dinputs = self.dinputs / samples

class Activation_softmax:
    
    def forward(self, inputs):

        exp_values = np.exp(inputs - np.max(inputs, axis = 1, keepdims = True))
        probabilities = exp_values/np.sum(exp_values, axis = 1, keepdims = True)
        self.output = probabilities

    def backward(self, dvalues):
        self.dinputs = np.empty_like(dvalues)

        for index, (single_output, single_dvalues) in enumerate(zip(self.output, dvalues)):
            single_output = single_output.reshape(-1,1)

            jacobian_matrix = np.diagflat(single_output) - np.dot(single_output, single_output.T)

            #calculate sample-wise gradient
            #and add it to the array of sample gradients
            self.dinputs[index] = np.dot(jacobian_matrix, single_dvalues)

class Activation_Softmax_Loss_CategoricalCrossentropy():

    def __init__(self):
        self.activation = Activation_softmax()
        self.loss = Loss_CategoricalCrossentropy()

    # Forward pass
    def forward(self, inputs, y_true):
        #output layer activation function
        self.activation.forward(inputs)

        #set the output
        self.output = self.activation.output

        #calculate and return loss value
        return self.loss.calculate(self.output, y_true)
    
    def backward(self, dvalues, y_true):

        samples = len(dvalues)

        if len(y_true.shape) == 2:
            y_true = np.argmax(y_true, axis = 1)

        self.dinputs = dvalues.copy()
        self.dinputs[range(samples), y_true] -= 1
        self.dinputs = self.dinputs/samples


In [ ]:
X, y = vertical_data(samples = 100, classes = 3)

# Dense layer with 2 input and 64 output values (64 neurons).
dense1 = Dense_layer(2,64)

activation1 = Activation_Relu()

#Create second dense layer that take output from previous layer hence 3 inputs and 3 output values
dense2 = Dense_layer(64,3)

#We are now using new class to calculate final softmax activation and loss calculation.
loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()

In [ ]:
#Create optimizer
optimizer = Optimizer_Adagrad(decay = 1e-4)

for epoch in range(10001):
    dense1.forward(X)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)

    loss = loss_activation.forward(dense2.output, y)
    predictions = np.argmax(loss_activation.output, axis = 1)
    if len(y.shape) == 2:
        y = np.argmax(y, axis = 1)
    accuracy = np.mean(predictions == y)

    if not epoch % 100:
        print(f'epoch: {epoch}, ' +
              f'acc: {accuracy:.3f}, ' +
              f'loss: {loss:.3f}')
        
    loss_activation.backward(loss_activation.output, y)
    dense2.backward(loss_activation.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    optimizer.pre_update_params()
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    optimizer.post_update_params()

AdaGrad also worked well but we got better accuracy with SGD with momentum.

## RMSProp

Root mean square propogation, calculates an adaptive learning rate per parameter. Unlike Adagrad, RMSProp can allow effective learning rate to increase or decrease.

$cache = rho * cache + (1-rho) * gradient^2$

RMSProp adds a mechanism similar to momentum (the second part) but also adds a per-parameter adaptive learning rate (the first part).

* Each update contains a part of the cache and updates it with a fraction of the new, squared, gradients. 
* The pre-parameter learning rate can either fall or rise, depending on the last updates and current gradient.

* rho is a hyperparameter, is a cache memory decay rate. rho controls how much of previous cache being remembered.

* When a large gradient is encountered, the weight update can be manipulated such as effective learning rate is scaled down. Similarly, when small gradient is encountered, the weight update can be manipulated such as effective learning rate is scaled up.

* The weights will not be permanently decaying.

* However, the model can still find itself stuck in a local minimum.

In [ ]:
class Optimizer_RMSProp:

    def __init__(self, learning_rate = 1.0, decay = 0, epsilon = 1e-7, rho = 0.9):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay = decay
        self.iterations = 0
        self.epsilon = epsilon
        self.rho = rho

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate * (1/(1+ self.decay * self.iterations))


    #update parameters
    def update_params(self, layer):

        if not hasattr(layer, 'weight_cache'):
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_cache = np.zeros_like(layer.biases)

        # Update cache with squared current gradients
        layer.weight_cache += self.rho * layer.weight_cache + (1 - self.rho) * layer.dweights ** 2
        layer.bias_cache += self.rho * layer.bias_cache + (1 - self.rho) * layer.dbiases ** 2

        layer.weights += -self.current_learning_rate * layer.dweights / (np.sqrt(layer.weight_cache) + self.epsilon)
        layer.biases += -self.current_learning_rate * layer.dbiases / (np.sqrt(layer.bias_cache) + self.epsilon)

    def post_update_params(self):
        self.iterations += 1

In [ ]:
optimizer = Optimizer_RMSProp(decay = 1e-4)

for epoch in range(10001):
    dense1.forward(X)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)

    loss = loss_activation.forward(dense2.output, y)
    predictions = np.argmax(loss_activation.output, axis = 1)
    if len(y.shape) == 2:
        y = np.argmax(y, axis = 1)
    accuracy = np.mean(predictions == y)

    if not epoch % 100:
        print(f'epoch: {epoch}, ' +
              f'acc: {accuracy:.3f}, ' +
              f'loss: {loss:.3f}')
        
    loss_activation.backward(loss_activation.output, y)
    dense2.backward(loss_activation.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    optimizer.pre_update_params()
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    optimizer.post_update_params()

The results didn't get any better.

In [ ]:
optimizer = Optimizer_RMSProp(decay = 1e-5, rho = 0.999, learning_rate=0.02)

for epoch in range(10001):
    dense1.forward(X)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)

    loss = loss_activation.forward(dense2.output, y)
    predictions = np.argmax(loss_activation.output, axis = 1)
    if len(y.shape) == 2:
        y = np.argmax(y, axis = 1)
    accuracy = np.mean(predictions == y)

    if not epoch % 100:
        print(f'epoch: {epoch}, ' +
              f'acc: {accuracy:.3f}, ' +
              f'loss: {loss:.3f}')
        
    loss_activation.backward(loss_activation.output, y)
    dense2.backward(loss_activation.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    optimizer.pre_update_params()
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    optimizer.post_update_params()

## Adam (Adaptive Momentum)

Adam is built atop RMSProp, with the momentum concept from SGD added back in. Instead of applying current gradients, we're going to apply momentums like in the SGD optimizer with momentum, then apply a per-weight adaptive learning rate with the cache as done in RMSProp.

We will also add a bias correction mechanism, compensating for the initial zeroes values before they warm up with initial steps.

The learning rate is scaled as :-

$v_{t+1} = \beta_2 v_t + (1-\beta_2)g(w_t)^2$
which is like RMSProp but the gradient jump is parallel to another term m, which is :-

$m_{t+1} = \beta_1 m_t + (1-\beta_1)g(w_t)$
which mimics the classical momentum 
$v_{t+1} = \rho v_t - \eta g(w_t)$.

Update rule :-
$w_{t+1} = w_t - \frac{\eta}{\epsilon + \sqrt{v_{t+1}}} m_{t+1}$

In [ ]:
class Optimizer_Adam:

    def __init__(self, learning_rate = 0.001, decay = 0, epsilon = 1e-7, beta_1 = 0.9, beta_2 = 0.999):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay = decay
        self.iterations = 0
        self.epsilon = epsilon
        self.beta_1 = beta_1
        self.beta_2 = beta_2

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate * (1/(1+ self.decay * self.iterations))


    #update parameters
    def update_params(self, layer):

        if not hasattr(layer, 'weight_cache'):
            layer.weight_momentums = np.zeros_like(layer.weights)
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_momentums = np.zeros_like(layer.biases)
            layer.bias_cache = np.zeros_like(layer.biases)

        #Update moementum with current gradients
        layer.weight_momentums = self.beta_1 * layer.weight_momentums + (1 - self.beta_1) * layer.dweights
        layer.bias_momentums = self.beta_1 * layer.bias_momentums + (1 - self.beta_1) * layer.dbiases

        weight_momentums_corrected = layer.weight_momentums / (1 - self.beta_1 ** (self.iterations + 1))
        bias_momentums_corrected = layer.bias_momentums / (1 - self.beta_1 ** (self.iterations + 1))

        # Update cache with squared current gradients
        layer.weight_cache = self.beta_2 * layer.weight_cache + (1 - self.beta_2) * layer.dweights ** 2
        layer.bias_cache += self.beta_2 * layer.bias_cache + (1 - self.beta_2) * layer.dbiases ** 2

        #Get corrected cache
        weight_cache_corrected = layer.weight_cache / (1 - self.beta_2 ** (self.iterations + 1))
        bias_cache_corrected = layer.bias_cache / (1 - self.beta_2 ** (self.iterations + 1))

        layer.weights += -self.current_learning_rate * weight_momentums_corrected / (np.sqrt(weight_cache_corrected) + self.epsilon)
        layer.biases += -self.current_learning_rate * bias_momentums_corrected / (np.sqrt(bias_cache_corrected) + self.epsilon)

    def post_update_params(self):
        self.iterations += 1

In [ ]:
optimizer = Optimizer_Adam(decay = 5e-7, learning_rate=0.05)

for epoch in range(10001):
    dense1.forward(X)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)

    loss = loss_activation.forward(dense2.output, y)
    predictions = np.argmax(loss_activation.output, axis = 1)
    if len(y.shape) == 2:
        y = np.argmax(y, axis = 1)
    accuracy = np.mean(predictions == y)

    if not epoch % 100:
        print(f'epoch: {epoch}, ' +
              f'acc: {accuracy:.3f}, ' +
              f'loss: {loss:.3f}, ' +
              f'lr: {optimizer.current_learning_rate}')
        
    loss_activation.backward(loss_activation.output, y)
    dense2.backward(loss_activation.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    optimizer.pre_update_params()
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    optimizer.post_update_params()

One optimizer can't always perform better than the other.

We will cover choosing various hyperparameters (such as the learning rate) when training, but a
general starting learning rate for SGD is 1.0, with a decay down to 0.1. For Adam, a good starting
LR is 0.001 (1e-3), decaying down to 0.0001 (1e-4). Different problems may require different
values here, but these are decent to start.